In [ ]:
import os
import sys
import json
import itertools
import sklearn
import re
from sklearn import tree
from sklearn import ensemble
from sklearn import linear_model
from sklearn import neural_network
from sklearn import svm
from scipy.stats import norm
from sklearn.metrics import confusion_matrix,roc_curve,auc
import random
import numpy as np
import pandas as pd
import math
import glob
import matplotlib

import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from scipy.optimize import curve_fit

In [ ]:

#### INPUTS
path_to_score_files = [
    "final_filter.sc",
]

# Leave this blank if you don't have one yet
path_to_fragment_quality_file = "h5_database.h5"



In [ ]:


dfs = []
for file in path_to_score_files:
    df = pd.read_csv(file, sep="\s+")
    df['file'] = file
    dfs.append(df)
    
pilot_df = pd.concat(dfs)
n = len(pilot_df)+1
nlist = range(1,n)
pilot_df['id'] = nlist
#pilot_df["description"] = pilot_df["description"] + 'a'

print("Number of designs: ", len(pilot_df))

In [ ]:
# The format here is:
# Name of the score in pilot, cut value, higher better, name in predictor, is integer score-term

terms_and_cuts = {
    'interface_buried_sasa':    [1400,  True,         "interface_buried_sasa_fr1", False],
    'ddg':                      [ -28, False,                           "ddg_fr1", False],
    'interface_sc':             [0.6,  True,                  "interface_sc_fr1", False],
#     'ddg_hydrophobic':          [ -18, False,               "ddg_hydrophobic_fr1", False],
    'score_per_res' :     [-2.6, False,           "score_per_res_fr1", False],
    'interface_sc_median_dist': [ 0.7, False,  "interface_sc_for_fr1_median_dist", False],
#     'fa_atr_pocket':            [-5,  False,         "fa_atr_pocket_fr1", False],
#   'sspred_overall':           [0.85,  True,                "sspred_overall_fr1", False],
 #   'mismatch_probability':     [0.3,  False,          "mismatch_probability_fr1", False],
    'contact_molecular_surface':[ 450,  True,     "contact_molecular_surface_fr1", False],
   'vbuns':  [   2, False,       "vbuns5.0_heavy_ball_1.1D_fr1",  True],
   'sbuns':  [  2, False,       "sbuns5.0_heavy_ball_1.1D_fr1",  True],
   'contact_MS':  [350,  True,  "contact_MS_fr1", False],
   'contact_patch':  [300, True,  "contact_patch_fr1", False],
}


score_df = pilot_df

# Filter all the terms and print the thresholds
ok_terms = []
for pilot_term in terms_and_cuts:
    cut, good_high, term, is_int = terms_and_cuts[pilot_term]
    ok_term = pilot_term.replace("_pilot", "") + "_ok"
    if ( good_high ):
        score_df[ok_term] = score_df[pilot_term] >= cut
    else:
        score_df[ok_term] = score_df[pilot_term] <= cut
    
    ok_terms.append(ok_term)
    
    print("%30s: %6.2f"%(pilot_term, cut))

# Print the pass rates for each term
print("")
score_df['orderable'] = True
for ok_term in ok_terms:
    score_df['orderable'] &= score_df[ok_term]
    print("%30s: %5.0f%% pass-rate"%(ok_term.replace("_ok", ""), float (score_df[ok_term].sum()) / len(score_df) * 100))

 
# print the overall pass rate   
subdf = score_df
print("Orderable: %i   -- %.2f%%"%(subdf['orderable'].sum(), (100*subdf['orderable'].sum() / len(subdf))))


In [ ]:
# Dump the tags for the Orderable designs from the pilot

out_folder = "final_filtering/"

for file in pilot_df['file'].unique():
    subdf = pilot_df[pilot_df['file'] == file]
    
    can_order = subdf[subdf['orderable']]
    
    final = "" if path_to_fragment_quality_file == "" else "_final"
    out_file = os.path.join(out_folder, file.replace(".", "_").replace("/", "_").lstrip("_") + 
                            final + "_45876.list")
    
    print("Input file: %s"%(file))
    print("   %8i / %8i orderable -- %.2f%%"%(len(can_order), len(subdf), float(len(can_order))/len(subdf)*100))
    #print("   Tags written to: "%(out_file))
    with open(out_file, "w") as f:
        for idx, row in can_order.iterrows():
            f.write("%s\n"%row['description'])